In [ ]:
# MDS650_v260807_Evapotranspiration.ipynb
"""
Extracts the biweekly actual evapotranspiration (ET) series for a point,
using dual source in Earth Engine:

1. MODIS/061/MOD16A2GF (Gap-Filled) - primary source, with gap filling,
   but "year-end gap-filled": the current year may have NO data at all
   until it closes.
2. MODIS/061/MOD16A2 (non gap-filled) - fallback source, near real-time,
   for biweekly periods where the primary source hasn't published yet.
   Without gap filling, so individual composites may still be missing
   due to persistent cloud cover.

HOW TO READ THE RESULT
-----------------------
et_total_mm    Total water lost through evaporation + transpiration in
               that biweekly period (based on the current vegetation of
               the pixel, not necessarily the crop you plan to plant -
               see note below).
source         Which product the data came from: 'GF', 'no-GF', or
               'no_data' if neither source had a record for that
               biweekly period.

USE IN WATER BALANCE: together with precipitation_profile.py (input)
and soil_hydraulics.py (storage capacity), this closes the balance:
    Δstorage = precipitation - ET - surplus (when exceeding AWC)

IMPORTANT LIMITATION: Actual ET reflects the EXISTING vegetation in the
pixel today, not the crop you plan to plant (which may consume more or
less water). It serves as an indicative baseline for deciding whether
installed irrigation capacity is needed, not as exact system sizing.

TECHNICAL LIMITATION (composites): both products come in 8-day composites
that don't align exactly with the biweekly period boundaries (1-15,
16-end of month) -> there may be a small shift at the boundaries. For
the purpose of detecting general deficit, this does not affect the
conclusion.

CITATIONS (methodology):
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration Gap-Filled 8-Day L4 Global 500m SIN Grid V061
[Data set]. NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2GF.061
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration 8-Day L4 Global 500m SIN Grid V061 [Data set].
NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2.061
"""
from datetime import date, datetime
from pathlib import Path
import pandas as pd
import ee

from period_utils import build_biweekly_periods
ee.Initialize()


def get_et_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    Extracts biweekly actual evapotranspiration statistics using a dual-source
    approach with automatic fallback from MOD16A2GF to MOD16A2.
    
    Priority logic:
    1. Try MOD16A2GF (gap-filled, more complete) first
    2. Fall back to MOD16A2 (non gap-filled, near real-time) if GF unavailable
    3. Mark as 'no_data' if neither source has coverage
    
    MOD16 ET values are scaled: raw value x 0.1 = real mm accumulated in the composite.
    The scale factor is applied immediately when loading the collections.
    
    Args:
        lat: Latitude of the point in degrees
        lon: Longitude of the point in degrees
        start_date: Start date for the analysis period (str "YYYY-MM-DD")
        end_date: End date for the analysis period (str "YYYY-MM-DD");
                  None defaults to today
    
    Returns:
        pandas.DataFrame: Biweekly ET statistics with columns:
                          period_start, period_end, label, et_total_mm, source
    """
    # Parse date strings to Python date objects
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    # Generate the list of complete biweekly periods within the date range
    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} biweekly periods to process, from {start} to {end}")

    # Create Earth Engine point geometry for the extraction location
    point = ee.Geometry.Point([lon, lat])

    # Load primary source: MOD16A2GF (Gap-Filled)
    # Apply scale factor (0.1) immediately to convert to real mm
    # Preserve system:time_start property for correct date filtering
    col_gf = (
        ee.ImageCollection('MODIS/061/MOD16A2GF')
        .select('ET')
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )
    
    # Load fallback source: MOD16A2 (non gap-filled, near real-time)
    # Apply the same scale factor and property preservation
    col_fallback = (
        ee.ImageCollection('MODIS/061/MOD16A2')
        .select('ET')
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )

    # Build the list of periods as an ee.List of dictionaries for server-side mapping
    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        """
        Server-side function to compute ET statistics for a single period
        using dual-source logic with automatic fallback.
        This runs on Google Earth Engine servers, not locally.
        
        Priority: MOD16A2GF (gap-filled) > MOD16A2 (near real-time) > no_data
        
        Args:
            period: ee.Dictionary with keys 'label', 'start', 'end'
        
        Returns:
            ee.Feature with computed ET total and source indicator
        """
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        # Filter both collections to the current period
        filtered_gf = col_gf.filterDate(p_start, p_end)
        filtered_fb = col_fallback.filterDate(p_start, p_end)

        # Check if each source has data available for this period
        has_gf = filtered_gf.size().gt(0)
        has_fb = filtered_fb.size().gt(0)

        # Build images for each source with appropriate source labels
        # If data exists: sum all composites and tag with source identifier
        # If no data: create a self-masked image (returns NaN) tagged as 'no_data'
        img_gf = ee.Image(ee.Algorithms.If(
            has_gf,
            filtered_gf.sum().rename('et_total_mm').set('source', 'GF'),
            ee.Image.constant(0).rename('et_total_mm').selfMask().set('source', 'no_data')
        ))
        img_fb = ee.Image(ee.Algorithms.If(
            has_fb,
            filtered_fb.sum().rename('et_total_mm').set('source', 'no-GF'),
            ee.Image.constant(0).rename('et_total_mm').selfMask().set('source', 'no_data')
        ))

        # Prioritize GF (gap-filled); if unavailable, use the fallback
        # (which may itself end up as 'no_data' if no data exists there either)
        final_img = ee.Image(ee.Algorithms.If(has_gf, img_gf, img_fb))

        # Extract the pixel value at the point location
        stats = final_img.reduceRegion(
            reducer=ee.Reducer.first(),  # Take the first (only) pixel value at the point
            geometry=point,
            scale=500,  # MOD16 native resolution (500m)
            maxPixels=1e9
        )

        # Return as a Feature with computed statistics, period metadata, and source
        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('period_start', p_start.format('YYYY-MM-dd'))
            .set('period_end', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
            .set('source', final_img.get('source'))
        )

    # Map the compute_period function over all periods (server-side execution)
    features = ee.FeatureCollection(ee_periods.map(compute_period))
    
    # Single network call to retrieve ALL periods' results at once
    result = features.getInfo()

    # Parse the Earth Engine response into a list of dictionaries
    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'period_start': props.get('period_start'),
            'period_end': props.get('period_end'),
            'label': props.get('label'),
            # 'lat': lat,   # Uncomment if you want coordinates in the output
            # 'lon': lon,   # Uncomment if you want coordinates in the output
            'et_total_mm': round(props.get('et_total_mm'), 2) if props.get('et_total_mm') is not None else None,
            'source': props.get('source'),
        })

    # Create DataFrame and sort chronologically by period start date
    df = pd.DataFrame(rows)
    df['period_start'] = pd.to_datetime(df['period_start'])
    df = df.sort_values('period_start').reset_index(drop=True)

    # Check for biweekly periods where neither source had data
    null_rows = df['et_total_mm'].isna().sum()
    if null_rows > 0:
        first_nulls = df[df['et_total_mm'].isna()]['label'].tolist()
        print(f"[WARNING] {null_rows} biweekly period(s) without data in any "
              f"source (neither GF nor no-GF): {first_nulls}")

    return df


def save_et_profile(df, out_prefix="et_biweekly", output_dir="../databases"):
    """
    Saves the evapotranspiration series with a timestamp in the filename:
    {out_prefix}-vYYMMDDHHMMSS.csv (same pattern as the rest of the pipeline)
    
    Args:
        df: DataFrame from get_et_biweekly()
        out_prefix: Base name for the output CSV file
        output_dir: Directory where the CSV is saved (created if it doesn't exist)
    
    Returns:
        pathlib.Path: Path to the saved CSV file
    """
    # Generate a unique filename with timestamp
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    # Save to CSV without the default pandas index
    df.to_csv(out_path, index=False)
    print(f"CSV saved to {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Test point: Sugarcane field in Queensland, Australia
    Latitude, Longitude =-19.689669877950884,147.22717515914223
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    # Extract biweekly ET data from 2016 onwards using dual-source approach
    df = get_et_biweekly(Latitude, Longitude, start_date="2016-01-01")
    
    # Save the ET profile to CSV
    out_path = save_et_profile(df)
    
    # Display the first 10 rows for inspection
    print(df.head(10))

[DEBUG] 254 biweekly periods to process, from 2016-01-01 to 2026-08-10
CSV saved to ../databases/et_biweekly-v260810172856.csv (254x5)
  period_start  period_end       label  et_total_mm source
0   2016-01-01  2016-01-15  2016-01_Q1         25.8     GF
1   2016-01-16  2016-01-31  2016-01_Q2         40.4     GF
2   2016-02-01  2016-02-15  2016-02_Q1         51.7     GF
3   2016-02-16  2016-02-29  2016-02_Q2         41.6     GF
4   2016-03-01  2016-03-15  2016-03_Q1         45.3     GF
5   2016-03-16  2016-03-31  2016-03_Q2         40.1     GF
6   2016-04-01  2016-04-15  2016-04_Q1         29.0     GF
7   2016-04-16  2016-04-30  2016-04_Q2         25.5     GF
8   2016-05-01  2016-05-15  2016-05_Q1         11.0     GF
9   2016-05-16  2016-05-31  2016-05_Q2         25.1     GF
